# Homework 1 – Designing DB Tables  
**Project:** Film Festival Management – Organizes film submissions, screening schedules, venues, and juror ratings for a festival.  

This notebook builds the schema, constraints, and inserts sample data.


# Part A – Context & Sequence

In [ ]:
-- Part A Context (login)
USE ROLE ROLE_MALLARD;
USE WAREHOUSE ANIMAL_TASK_WH;
USE DATABASE DB_MALLARD;
USE DB_MALLARD.HOMEWORKS;

-- One sequence used by all non-junction surrogate PKs
CREATE OR REPLACE SEQUENCE HW1_SEQ START = 1 INCREMENT = 1;

# Part B – Core Entity Tables

In [ ]:
CREATE OR REPLACE TABLE Festivals (
  festival_id NUMBER PRIMARY KEY DEFAULT HW1_SEQ.NEXTVAL,
  name        VARCHAR(150) NOT NULL,
  year        NUMBER       NOT NULL,
  start_date  DATE         NOT NULL,
  end_date    DATE         NOT NULL,
  is_virtual  BOOLEAN      NOT NULL
);

In [ ]:
CREATE OR REPLACE TABLE Films (
  film_id          NUMBER PRIMARY KEY DEFAULT HW1_SEQ.NEXTVAL,
  title            VARCHAR(200) NOT NULL,
  runtime_minutes  NUMBER       NOT NULL,
  language         VARCHAR(60),
  country          VARCHAR(60),
  genre            VARCHAR(60),
  premiere_status  VARCHAR(60)
);

In [ ]:
CREATE OR REPLACE TABLE Directors (
  director_id NUMBER PRIMARY KEY DEFAULT HW1_SEQ.NEXTVAL,
  full_name   VARCHAR(120) NOT NULL,
  country     VARCHAR(60),
  email       VARCHAR(120)
);

In [ ]:
CREATE OR REPLACE TABLE Venues (
  venue_id          NUMBER PRIMARY KEY DEFAULT HW1_SEQ.NEXTVAL,
  name              VARCHAR(120) NOT NULL,
  address           VARCHAR(200),
  city              VARCHAR(80),
  capacity          NUMBER       NOT NULL,
  has_4k_projection BOOLEAN      NOT NULL
);

In [ ]:
CREATE OR REPLACE TABLE Jurors (
  juror_id       NUMBER PRIMARY KEY DEFAULT HW1_SEQ.NEXTVAL,
  full_name      VARCHAR(120) NOT NULL,
  expertise_area VARCHAR(80),
  email          VARCHAR(120)
);

# Part C – Relationship / Junction Tables

In [ ]:
-- Films ↔ Directors (M:N)
CREATE OR REPLACE TABLE FilmDirectors (
  film_id     NUMBER NOT NULL,
  director_id NUMBER NOT NULL,
  is_primary  BOOLEAN NOT NULL,
  CONSTRAINT pk_film_directors PRIMARY KEY (film_id, director_id),
  CONSTRAINT fk_fd_film FOREIGN KEY (film_id) REFERENCES Films(film_id),
  CONSTRAINT fk_fd_dir  FOREIGN KEY (director_id) REFERENCES Directors(director_id)
);

In [ ]:
-- Films ↔ Festivals (M:N) via Submissions
CREATE OR REPLACE TABLE Submissions (
  submission_id NUMBER PRIMARY KEY DEFAULT HW1_SEQ.NEXTVAL,
  film_id       NUMBER NOT NULL,
  festival_id   NUMBER NOT NULL,
  submitted_at  DATETIME NOT NULL,
  status        VARCHAR(40) NOT NULL,
  fee_usd       FLOAT,
  is_late       BOOLEAN NOT NULL,
  CONSTRAINT fk_sub_film     FOREIGN KEY (film_id)     REFERENCES Films(film_id),
  CONSTRAINT fk_sub_festival FOREIGN KEY (festival_id)  REFERENCES Festivals(festival_id),
  CONSTRAINT uq_sub UNIQUE (film_id, festival_id)   -- enforces one submission per film+festival
);

In [ ]:
-- Jurors ↔ Films (M:N), scoped by Festival via Ratings
CREATE OR REPLACE TABLE Ratings (
  rating_id   NUMBER PRIMARY KEY DEFAULT HW1_SEQ.NEXTVAL,
  juror_id    NUMBER NOT NULL,
  film_id     NUMBER NOT NULL,
  festival_id NUMBER NOT NULL,
  score       FLOAT  NOT NULL,
  comments    VARCHAR(500),
  CONSTRAINT fk_rat_juror    FOREIGN KEY (juror_id)    REFERENCES Jurors(juror_id),
  CONSTRAINT fk_rat_film     FOREIGN KEY (film_id)     REFERENCES Films(film_id),
  CONSTRAINT fk_rat_festival FOREIGN KEY (festival_id) REFERENCES Festivals(festival_id),
  CONSTRAINT uq_rat UNIQUE (juror_id, film_id, festival_id) -- one rating per juror/film/festival
);

In [ ]:

CREATE OR REPLACE TABLE Screenings (
  screening_id NUMBER PRIMARY KEY DEFAULT HW1_SEQ.NEXTVAL,
  festival_id  NUMBER   NOT NULL,
  film_id      NUMBER   NOT NULL,
  venue_id     NUMBER   NOT NULL,
  start_time   DATETIME NOT NULL,
  is_premiere  BOOLEAN  NOT NULL,
  CONSTRAINT fk_scr_festival FOREIGN KEY (festival_id) REFERENCES Festivals(festival_id),
  CONSTRAINT fk_scr_film     FOREIGN KEY (film_id)     REFERENCES Films(film_id),
  CONSTRAINT fk_scr_venue    FOREIGN KEY (venue_id)    REFERENCES Venues(venue_id)
);

In [ ]:
CREATE OR REPLACE TABLE Awards (
  award_id         NUMBER PRIMARY KEY DEFAULT HW1_SEQ.NEXTVAL,
  festival_id      NUMBER NOT NULL,
  film_id          NUMBER NOT NULL,
  category         VARCHAR(100) NOT NULL,
  prize_amount_usd FLOAT,
  CONSTRAINT fk_aw_festival FOREIGN KEY (festival_id) REFERENCES Festivals(festival_id),
  CONSTRAINT fk_aw_film     FOREIGN KEY (film_id)     REFERENCES Films(film_id)
);

# Part D – Sample Inserts

In [ ]:
-- Festivals (2)
INSERT INTO Festivals (name, year, start_date, end_date, is_virtual) VALUES
 ('Austin International Film Fest', 2025, '2025-10-10', '2025-10-17', FALSE),
 ('Austin International Film Fest', 2026, '2026-10-09', '2026-10-16', TRUE);

In [ ]:
-- Films (3)
INSERT INTO Films (title, runtime_minutes, language, country, genre, premiere_status) VALUES
 ('River’s Edge', 104, 'English', 'USA', 'Drama', 'US Premiere'),
 ('Silent Frames', 89, 'French', 'France', 'Documentary', 'World Premiere'),
 ('Desert Wind', 117, 'Arabic', 'Morocco', 'Adventure', 'Regional Premiere');

In [ ]:
-- Directors (3)
INSERT INTO Directors (full_name, country, email) VALUES
 ('Maya Lee', 'USA', 'maya.lee@example.com'),
 ('Omar Haddad', 'Morocco', 'omar.h@example.com'),
 ('Claire Dubois', 'France', 'claire.d@example.fr');

In [ ]:
-- Venues (2)
INSERT INTO Venues (name, address, city, capacity, has_4k_projection) VALUES
 ('Paramount Theatre', '713 Congress Ave', 'Austin', 1200, TRUE),
 ('Stateside at the Paramount', '719 Congress Ave', 'Austin', 300, FALSE);

In [ ]:
-- Jurors (3)
INSERT INTO Jurors (full_name, expertise_area, email) VALUES
 ('Anita Rao', 'Cinematography', 'anita.rao@example.com'),
 ('David Kim', 'Screenwriting', 'david.kim@example.com'),
 ('Lucia Gomez', 'Directing', 'lucia.g@example.com');

In [ ]:
-- FilmDirectors (link)
INSERT INTO FilmDirectors (film_id, director_id, is_primary)
SELECT f.film_id, d.director_id, TRUE
FROM Films f JOIN Directors d
  ON f.title='River’s Edge' AND d.full_name='Maya Lee'
UNION ALL
SELECT f.film_id, d.director_id, TRUE
FROM Films f JOIN Directors d
  ON f.title='Silent Frames' AND d.full_name='Claire Dubois'
UNION ALL
SELECT f.film_id, d.director_id, TRUE
FROM Films f JOIN Directors d
  ON f.title='Desert Wind' AND d.full_name='Omar Haddad';

In [ ]:
-- Submissions (≥2 for the 2025 festival)
INSERT INTO Submissions (film_id, festival_id, submitted_at, status, fee_usd, is_late)
SELECT (SELECT film_id FROM Films WHERE title='River’s Edge'),
       (SELECT festival_id FROM Festivals WHERE year=2025),
       '2025-04-15 14:30:00', 'Selected', 45.0, FALSE
UNION ALL
SELECT (SELECT film_id FROM Films WHERE title='Silent Frames'),
       (SELECT festival_id FROM Festivals WHERE year=2025),
       '2025-04-18 09:05:00', 'In Review', 55.0, FALSE
UNION ALL
SELECT (SELECT film_id FROM Films WHERE title='Desert Wind'),
       (SELECT festival_id FROM Festivals WHERE year=2025),
       '2025-03-01 22:10:00', 'Rejected', 30.0, TRUE;

In [ ]:
-- Screenings (≥2 for the 2025 festival)
INSERT INTO Screenings (festival_id, film_id, venue_id, start_time, is_premiere)
SELECT (SELECT festival_id FROM Festivals WHERE year=2025),
       (SELECT film_id FROM Films WHERE title='River’s Edge'),
       (SELECT venue_id FROM Venues WHERE name='Paramount Theatre'),
       '2025-10-11 19:00:00', TRUE
UNION ALL
SELECT (SELECT festival_id FROM Festivals WHERE year=2025),
       (SELECT film_id FROM Films WHERE title='Silent Frames'),
       (SELECT venue_id FROM Venues WHERE name='Stateside at the Paramount'),
       '2025-10-12 16:00:00', TRUE
UNION ALL
SELECT (SELECT festival_id FROM Festivals WHERE year=2025),
       (SELECT film_id FROM Films WHERE title='Desert Wind'),
       (SELECT venue_id FROM Venues WHERE name='Paramount Theatre'),
       '2025-10-14 21:15:00', FALSE;

In [ ]:
-- Ratings (multiple jurors/films for 2025)
INSERT INTO Ratings (juror_id, film_id, festival_id, score, comments)
SELECT (SELECT juror_id FROM Jurors WHERE full_name='Anita Rao'),
       (SELECT film_id FROM Films WHERE title='River’s Edge'),
       (SELECT festival_id FROM Festivals WHERE year=2025),
       8.7, 'Stunning visuals and pacing.'
UNION ALL
SELECT (SELECT juror_id FROM Jurors WHERE full_name='David Kim'),
       (SELECT film_id FROM Films WHERE title='River’s Edge'),
       (SELECT festival_id FROM Festivals WHERE year=2025),
       8.2, 'Strong character arcs.'
UNION ALL
SELECT (SELECT juror_id FROM Jurors WHERE full_name='Anita Rao'),
       (SELECT film_id FROM Films WHERE title='Silent Frames'),
       (SELECT festival_id FROM Festivals WHERE year=2025),
       7.5, 'Thoughtful but slow in parts.';

In [ ]:
-- Awards (≥2 for 2025)
INSERT INTO Awards (festival_id, film_id, category, prize_amount_usd)
SELECT (SELECT festival_id FROM Festivals WHERE year=2025),
       (SELECT film_id FROM Films WHERE title='River’s Edge'),
       'Best Film', 10000.0
UNION ALL
SELECT (SELECT festival_id FROM Festivals WHERE year=2025),
       (SELECT film_id FROM Films WHERE title='Silent Frames'),
       'Best Documentary', 5000.0;

In [ ]:
-- View tables & all rows from each table
-- SHOW TABLES IN SCHEMA HOMEWORKS;
-- SELECT * FROM Festivals;
-- SELECT * FROM Films;
-- SELECT * FROM Directors;
-- SELECT * FROM FilmDirectors;
-- SELECT * FROM Submissions;
-- SELECT * FROM Venues;
-- SELECT * FROM Screenings;
-- SELECT * FROM Jurors;
-- SELECT * FROM Ratings;
-- SELECT * FROM Awards;

# Part E – Cleanup (commented)

In [ ]:
-- Part E — Cleanup (KEEP COMMENTED)

-- DROP TABLE IF EXISTS Awards;
-- DROP TABLE IF EXISTS Screenings;
-- DROP TABLE IF EXISTS Ratings;
-- DROP TABLE IF EXISTS Submissions;
-- DROP TABLE IF EXISTS FilmDirectors;
-- DROP TABLE IF EXISTS Jurors;
-- DROP TABLE IF EXISTS Venues;
-- DROP TABLE IF EXISTS Directors;
-- DROP TABLE IF EXISTS Films;
-- DROP TABLE IF EXISTS Festivals;
-- DROP SEQUENCE IF EXISTS HW1_SEQ;